In [ ]:
import wfdb
import numpy as np
from utils.data import filter_data,normalize,split_signal

In [ ]:
for i in range(1, 54):
    file_id = f"bidmc{i:02d}"
    file_path = f"/root/tmp/PulseGUARD/data_bidmc/{file_id}"
    
    try:
        # Read the records
        record = wfdb.rdrecord(file_path)
        p_signals = record.p_signal
        sig_names = record.sig_name
        
        # Obtain signal index
        ecg_index = sig_names.index('II,')
        ppg_index = sig_names.index('PLETH,')
        
        # Extract the signal
        ecg_signal = p_signals[:, ecg_index]
        ppg_signal = p_signals[:, ppg_index]
        
        # Process the signal
        Fs = record.fs  # Obtain the actual sampling rate
        filtered_ecg = filter_data(ecg_signal, Fs)
        norm_ecg = normalize(filtered_ecg).flatten()
        norm_ppg = normalize(ppg_signal)
        
        # Splitting signals into segments
        ecg_segments = np.array(split_signal(norm_ecg))
        ppg_segments = np.array(split_signal(norm_ppg))
        
        # Save the results
        np.save(f'/root/tmp/PulseGUARD/data_gan_3s/ecg/{file_id}.npy', ecg_segments)
        np.save(f'/root/tmp/PulseGUARD/data_gan_3s/ppg/{file_id}.npy', ppg_segments)
        print(f"Successfully handled: {file_id}")
        
    except FileNotFoundError:
        print(f"The file does not exist.: {file_id}")
    except ValueError as ve:
        print(f"Signal index error: {file_id} - {str(ve)}")
    except Exception as e:
        print(f"An error occurred while processing{file_id}: {str(e)}")